# Stratified DFT Hessians

Open this notebook from GitHub and use a fresh GPU runtime. This environment pins GPU4PySCF 1.3.0, matching HORM, and runs neutral singlet wB97X/6-31G(d) calculations with the paper's SCF tolerances. Start with two cases to calibrate wall time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPOSITORY_URL = 'https://github.com/jiaxi98/OAReactDiff.git'
REPOSITORY_REF = 'agent/oa-failure-audit'
REPO = Path('/content/OAReactDiff')
OUTPUT_ROOT = Path('/content/drive/MyDrive/OAReactDiff/audit_outputs')
SCREEN_ROOT = OUTPUT_ROOT / 'horm_screen_generation_8x8_r2_j2'
SUBSET = SCREEN_ROOT / 'dft_subset.csv'
assert SUBSET.is_file(), f'Run 02_colab_horm_screen.ipynb first: missing {SUBSET}'
if not (REPO / '.git').is_dir():
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {REPOSITORY_REF} {REPOSITORY_URL} {REPO}
else:
    !git -C {REPO} fetch --depth 1 origin {REPOSITORY_REF}
    !git -C {REPO} checkout --detach FETCH_HEAD
!git -C {REPO} rev-parse HEAD
!cd {REPO} && bash experiments/oa_failure_audit/setup_colab_dft.sh

In [ ]:
MAMBA = '/usr/local/bin/micromamba'
ENV = 'oa-dft'
BENCHMARK_OUTPUT = SCREEN_ROOT / 'dft_benchmark_two'
!cd {REPO} && {MAMBA} run -n {ENV} python experiments/oa_failure_audit/run_dft_hessian.py \
    --subset {SUBSET} --output-dir {BENCHMARK_OUTPUT} --backend gpu4pyscf \
    --max-candidates 2 --resume

In [ ]:
import csv
with (BENCHMARK_OUTPUT / 'dft_results.csv').open() as handle:
    benchmark_rows = list(csv.DictReader(handle))
mean_seconds = sum(float(row['wall_seconds']) for row in benchmark_rows) / len(benchmark_rows)
print(f'Mean: {mean_seconds / 60:.1f} minutes/case')
print(f'Projected 96 cases: {mean_seconds * 96 / 3600:.1f} serial GPU-hours')

In [ ]:
RUN_FULL_SUBSET = False
DFT_OUTPUT = SCREEN_ROOT / 'dft_full'
if RUN_FULL_SUBSET:
    !cd {REPO} && {MAMBA} run -n {ENV} python experiments/oa_failure_audit/run_dft_hessian.py \
        --subset {SUBSET} --output-dir {DFT_OUTPUT} --backend gpu4pyscf --resume

## Gate IRC by DFT evidence

Only a stationary DFT index-1 point is immediately IRC-eligible. A nonstationary index-1 raw sample is routed through TS optimization and another Hessian first.

In [ ]:
if RUN_FULL_SUBSET:
    DFT_ENRICHED = SCREEN_ROOT / 'dft_subset_with_results.csv'
    !cd {REPO} && {MAMBA} run -n {ENV} python experiments/oa_failure_audit/merge_screening.py \
        --manifest {SUBSET} --screen dft={DFT_OUTPUT / 'dft_results.csv'} \
        --output {DFT_ENRICHED} --overwrite
    IRC_WORKLIST = SCREEN_ROOT / 'irc_worklist.csv'
    !cd {REPO} && {MAMBA} run -n {ENV} python experiments/oa_failure_audit/prepare_irc_worklist.py \
        --manifest {DFT_ENRICHED} --output {IRC_WORKLIST} --overwrite